In [12]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split, KFold, StratifiedKFold
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import classification_report
from cvxopt import matrix, solvers
import matplotlib.pyplot as plt
from SVM import *

In [2]:
df = pd.read_csv('dataset/ETHNICITY_CLASSIFICATION.csv')
df.head()

,feat_1,feat_2,feat_3,feat_4,feat_5,feat_6,feat_7,feat_8,feat_9,feat_10,...,feat_24,feat_25,feat_26,feat_27,feat_28,feat_29,feat_30,feat_31,feat_32,gt
0,0.502792,-0.349373,-0.068018,-0.627533,0.130331,0.373488,-0.491088,0.416753,-0.046255,0.325566,...,0.374539,-0.605601,0.579226,-0.119241,0.185425,-0.255194,0.705415,-0.027761,0.581276,0
1,0.741646,-0.240194,-0.006548,-0.639129,-0.059524,0.457087,-0.500733,0.345128,-0.040395,0.334946,...,0.353851,-0.651629,0.129014,-0.034723,0.392575,-0.289928,0.380891,-0.157109,0.742231,0
2,2.836617,1.256781,2.227900,-0.603728,0.200403,1.366685,-0.666864,1.156750,-0.004516,0.767839,...,0.265453,-0.762246,-0.331476,0.051758,1.255034,-0.529064,1.412283,1.182029,3.046791,0
3,2.576996,1.784919,3.166102,-0.539903,0.108954,1.773622,-0.708488,2.679638,-0.313469,1.817676,...,0.295006,-0.866968,2.203584,-0.428082,-0.368396,-0.398743,4.043780,2.281445,5.065996,0
4,1.614504,1.103544,0.755605,-0.561434,-0.551708,0.936586,-0.575693,0.533321,0.033991,0.980126,...,0.155448,-0.825574,-0.627675,0.060466,0.815617,-0.434781,0.040881,-0.152861,2.191638,0


In [28]:
target_values = list(range(3))
# Split features and target
df_sel = df[df['gt'].isin(target_values)]
X = df_sel.drop("gt", axis=1).values
y = df_sel["gt"].values


# Train/val split (80/20)
split = int(0.8 * len(X))
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# ----- 3. Istanzia e allena il tuo SVM con SMO -----
model = MultiSVM(C=1.0, kernel='Gaussian', gamma=0.5)

# Allenamento con il metodo SMO (fit_smo)
model.fit(X_train, y_train)

y_train_pred = model.predict(X_train)
y_test_pred = model.predict(X_test)

print("Accuracy on training set:\n", classification_report(y_train, y_train_pred))
print("\nAccuracy on test set:\n", classification_report(y_test, y_test_pred))

     pcost       dcost       gap    pres   dres
 0: -1.8139e+02 -2.5059e+03  1e+04  2e+00  9e-16
 1: -1.3009e+02 -1.4521e+03  1e+03  4e-02  6e-16
 2: -1.6616e+02 -3.7061e+02  2e+02  6e-03  7e-16
 3: -1.8415e+02 -2.3132e+02  5e+01  1e-03  4e-16
 4: -1.8960e+02 -2.0016e+02  1e+01  1e-04  3e-16
 5: -1.9106e+02 -1.9392e+02  3e+00  2e-05  3e-16
 6: -1.9151e+02 -1.9225e+02  7e-01  3e-06  3e-16
 7: -1.9166e+02 -1.9178e+02  1e-01  2e-15  3e-16
 8: -1.9168e+02 -1.9170e+02  1e-02  9e-15  3e-16
 9: -1.9169e+02 -1.9169e+02  7e-04  9e-15  3e-16
10: -1.9169e+02 -1.9169e+02  6e-05  5e-15  3e-16
Optimal solution found.
     pcost       dcost       gap    pres   dres
 0: -1.7438e+02 -2.5349e+03  1e+04  2e+00  9e-16
 1: -1.1985e+02 -1.4938e+03  2e+03  8e-02  6e-16
 2: -1.5460e+02 -3.7495e+02  2e+02  1e-02  9e-16
 3: -1.7460e+02 -2.2381e+02  5e+01  2e-03  5e-16
 4: -1.8020e+02 -1.9407e+02  1e+01  3e-04  3e-16
 5: -1.8201e+02 -1.8596e+02  4e+00  3e-15  3e-16
 6: -1.8252e+02 -1.8354e+02  1e+00  6e-15  3e-1

In [ ]:
from itertools import combinations

class MultiSVM(SVM):
    def __init__(self, method="OvR", *args, **kwargs):
        super().__init__(*args, **kwargs)
        self.models = []
        self.classes = None
        self.binary_model = None
        self.method = method
        if method == "OvO":
            self.pair_classes = []


    def fit(self, X, y):
        classes = np.unique(y)
        if self.method != "OvO" and self.method != "OvR":
            raise(ValueError('Please select "OvO" for One vs One Classification or "OvR" for One vs Rest Classification'))
        
        if len(classes) < 2 or len(classes) > 3:
            raise(ValueError("Please select 2 or 3 classes"))
        
        self.classes = np.array(classes)
        if len(classes) == 2:
            self.binary_model = SVM(C=self.C, kernel=self.kernel_type, gamma=self.gamma, p=self.p, tol=self.tol)
            self.binary_model.fit(X, y)
        elif len(classes) == 3 and self.method == "OvR":        
            for cls in classes:
                # One-vs-rest
                y_bin = np.where(y == cls, 1, -1)
                model = SVM(C=self.C, kernel=self.kernel_type, gamma=self.gamma, p=self.p, tol=self.tol)
                model.fit(X, y_bin)
                self.models[cls] = model
        elif len(classes) == 3 and self.method == "OvO":
            for cls1, cls2 in combinations(self.classes, 2):
                idx = np.where((y == cls1) | (y == cls2))[0]
                X_pair = X[idx]
                y_pair = y[idx]
                y_bin = np.where(y_pair == cls1, 1, -1)
                model = SVM(C=self.C, kernel=self.kernel_type, gamma=self.gamma, p=self.p, tol=self.tol)
                model.fit(X_pair, y_bin)
                self.models.append(model)
                self.pair_classes.append((cls1, cls2))
        return self

    def predict(self, X):
        if len(self.classes) == 2:
            return self.binary_model.predict(X)
        elif len(self.classes) == 3 and self.method == "OvR":
            # Decision Function for each model
            scores = np.column_stack([self.models[cls].decision_function(X) for cls in self.classes])
            # Return highest scoring model
            idx = np.argmax(scores, axis=1)
            return self.classes[idx]
        elif len(self.classes) == 3 and self.method == "OvO":
            votes = np.zeros((X.shape[0], len(self.classes)), dtype=int)
            for model, (cls1, cls2) in zip(self.models, self.pair_classes):
                pred = model.predict(X)
                for i, p in enumerate(pred):
                    if p == 1:
                        votes[i, np.where(self.classes == cls1)[0][0]] += 1
                    else:
                        votes[i, np.where(self.classes == cls2)[0][0]] += 1
            return self.classes[np.argmax(votes, axis=1)]

In [45]:
target_values = list(range(3))
# Split features and target
df_sel = df[df['gt'].isin(target_values)]
X = df_sel.drop("gt", axis=1).values
y = df_sel["gt"].values


# Train/val split (80/20)
split = int(0.8 * len(X))
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# ----- 3. Istanzia e allena il tuo SVM con SMO -----
model = MultiSVM(C=1.0, kernel='Gaussian', gamma=0.5, method="OvO")

# Allenamento con il metodo SMO (fit_smo)
model.fit(X_train, y_train)

y_train_pred = model.predict(X_train)
y_test_pred = model.predict(X_test)

print("Accuracy on training set:\n", classification_report(y_train, y_train_pred))
print("\nAccuracy on test set:\n", classification_report(y_test, y_test_pred))

     pcost       dcost       gap    pres   dres
 0: -1.3413e+02 -1.7470e+03  8e+03  2e+00  8e-16
 1: -9.9682e+01 -1.0344e+03  9e+02  5e-03  6e-16
 2: -1.2983e+02 -2.5363e+02  1e+02  6e-04  7e-16
 3: -1.4174e+02 -1.6835e+02  3e+01  8e-05  3e-16
 4: -1.4501e+02 -1.5044e+02  5e+00  8e-06  3e-16
 5: -1.4579e+02 -1.4720e+02  1e+00  1e-06  3e-16
 6: -1.4599e+02 -1.4651e+02  5e-01  8e-16  2e-16
 7: -1.4607e+02 -1.4616e+02  9e-02  4e-15  3e-16
 8: -1.4608e+02 -1.4610e+02  1e-02  2e-15  2e-16
 9: -1.4609e+02 -1.4609e+02  4e-04  2e-15  3e-16
10: -1.4609e+02 -1.4609e+02  8e-06  1e-14  3e-16
Optimal solution found.
     pcost       dcost       gap    pres   dres
 0: -1.3023e+02 -1.5061e+03  6e+03  2e+00  8e-16
 1: -1.0533e+02 -8.3918e+02  7e+02  3e-03  5e-16
 2: -1.3302e+02 -2.2767e+02  9e+01  4e-04  6e-16
 3: -1.4325e+02 -1.6602e+02  2e+01  7e-05  3e-16
 4: -1.4622e+02 -1.5126e+02  5e+00  1e-05  2e-16
 5: -1.4701e+02 -1.4817e+02  1e+00  1e-06  2e-16
 6: -1.4722e+02 -1.4746e+02  2e-01  8e-15  2e-1